## MODFLOW 6 Lab: Customizing for Flow, Transport and Geothermal Modeling

### 1.1 Hands-on: Flow Model 

In this session, you'll learn to:
- Build a simple groundwater flow model using MODFLOW 6.
- Visualize flow and results 
- Extend the model with dynamic pumping and river interactions.

We'll work step-by-step and feel free to modify parameters and rerun cells!

### Model description 

This groundwater model simulates the extraction of water of an aquifer.
#### Domain 
The model domain is a 2D single-layer grid consisting of 101 rows and columns, representing a 100 m × 100 m unconfined aquifer with a thickness of 30 m. 
#### Boundary conditions 
Boundary conditions include constant head boundaries on the left and right edges of the domain to simulate regional flow, with head values set to 25.0 m and 24.5 m, respectively. One extraction well is position in the top of the aquifer at cell x. 

### Goal description 

Regulate the model to extract the most amount of water possible while being regulated by the groundwater level threshold of 1 m. 

### Setup and Imports 

In [51]:
import flopy
import numpy as np
import matplotlib.pyplot as plt
import os

# Define workspace
workspace = os.path.join(os.getcwd(), "models/mf6/ws_example_1")
os.makedirs(workspace, exist_ok=True)
print("Workspace:", workspace)

Workspace: C:\Users\lucialabarca\re-run noteboks\pymf6-validation\src\notebooks\presentation_files\models/mf6/ws_example_1


### Build the MODFLOW 6 Base Flow Model 

## Start setting up the model with pymf6 

### Magic commands - auto reload of the model each time

In [52]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [53]:
%autoreload 2

### Import from pymf6tools the functions to run, get and visualize simulation results

In [54]:
from pathlib import Path 
from pymf6.mf6 import MF6
import pandas as pd 
from functools import partial 
import numpy as np 

from pymf6_tools.make_model import run_simulation, get_simulation
from pymf6_tools.base_model import make_model_data
from pymf6_tools.make_model import make_input, run_simulation, get_simulation
from pymf6_tools.plotting import show_heads, show_well_head, show_bcs

In [55]:
from pymf6_tools.plotting import (
show_heads, show_well_head, show_concentration, show_bcs, 
show_bot_elevations, show_river_stages, contour_bot_elevations, 
plot_spec_discharge)
import pymf6_tools

## Setup model path and name 

In [56]:
model_path = 'models/mf6/ws_example_1'
model_name = "ws_example_1"

### Run simulation

In [57]:
run_simulation(model_path, verbosity_level=0)

### Inpect model parameters 

In [58]:
sim = get_simulation(model_path, model_name)
ml = sim.get_model('gwf_' + model_name)
dis = ml.get_package('dis') 

In [59]:
dis.data_list

[,
 ,
 ,
 ,
 ,
 ,
 ,
 ,
 ,
 {internal}
 (1),
 {internal}
 (101),
 {internal}
 (101),
 {constant 1.0},
 {constant 1.0},
 {constant 0.0},
 {constant -30.0},
 ]

## Visualization of Input and Output - e.g. Boundary conditions, Heads and Contamination plume¶

#### Boundary Conditions
Note that you should change the "bc_names" according to the boundary conditions present in the simulation. For instance:
'chd' Constant-head boundary
'riv-1' River boundary

In [60]:
show_bcs?

Signature:
show_bcs(
    model_path,
    name,
    title='Boundary Conditions',
    bc_names=('chd', 'wel', 'riv'),
    show_grid=True,
)
Docstring: Show location of boundary conditions.
File:      c:\users\lucialabarca\re-run noteboks\pymf6-validation\.pixi\envs\default\lib\site-packages\pymf6_tools\plotting.py
Type:      function

### Groundwater level 

In [66]:
show_heads(model_path, model_name, show_wells=False, kstpkper=(10, 1), show_grid=False)

IndexError: list index out of range

In [ ]:
This graph shows the groundwater head distribution in the domain. 

🔍 What is kstpkper?
In MODFLOW 6, kstpkper = (kstp, kper) refers to:

kstp: Time step within a stress period

kper: Stress period number

FloPy tracks head and budget data per (kstp, kper). If you ask for a timestep combination that wasn't saved or doesn't exist, it throws this error.

✅ How to Find your kstper?
Check your simulation's stress period setup
From the model in this case 

perioddata=[(1.0, 1, 1.0), (300.0, 20, 1.0)]

There are:

Stress period 0: 1 time step

Stress period 1: 20 time steps

So valid kstpkper values:

For kper=0: only (0, 0)

For kper=1: (0, 1) to (19, 1)

Since we want the final time then is its (19, 1).

### Inpect MODFLOW 6 interactively 

pymf6 allows to access all MODFLOW 6 variables at run time. First, we import the class MF6:

In [67]:
from pymf6.mf6 import MF6

We use the model path 

In [69]:
sim_path = 'models/pymf6/ws_example_1'

In [70]:
mf6 = MF6(sim_path=sim_path)


The instance has an HTML representation with some meta data:

In [71]:
mf6

pymf6 version:,1.5.2
xmipy version:,1.3.1
modflowapi version:,0.3.0
ini file path:,C:\Users\lucialabarca\pymf6.ini
dll file path:,C:\Users\lucialabarca\mf6.6.2_win64\bin\libmf6.dll
MODFLOW version:,6.6.2


In [72]:
mf6.info 

pymf6 configuration data
pymf6 version: 1.5.2
xmipy version: 1.3.1
modflowapi version: 0.3.0
ini file path: C:\Users\lucialabarca\pymf6.ini
dll file path: C:\Users\lucialabarca\mf6.6.2_win64\bin\libmf6.dll
MODFLOW version: 6.6.2
MODFLOW Fortan variable documentation is NOT available.


In [73]:
mf6.simulation

modeltype,namefile,modelname
gwf6,gwf_ws_example_1.nam,GWF_WS_EXAMPLE_1


In [74]:
mf6.simulation.model_names

['GWF_WS_EXAMPLE_1']

The meta information is also available as a dictionary:

In [75]:
mf6.simulation.models_meta 

[{'modeltype': 'gwf6',
  'namefile': 'gwf_ws_example_1.nam',
  'modelname': 'GWF_WS_EXAMPLE_1'}]

This example has only one solution group:

In [79]:
mf6.simulation.solution_groups 

[Solution 1 
 1 packages
 60 variables.]

Time discretization:

In [81]:
mf6.simulation.TDIS

number of variables:,20


The names of the variables

In [83]:
mf6.simulation.TDIS.var_names

['PERTIMSAV',
 'TOPERTIM',
 'ENDOFSIMULATION',
 'TOTIMSAV',
 'PERTIM',
 'ITMUNI',
 'KSTP',
 'PERLEN',
 'TOTIM',
 'ENDOFPERIOD',
 'NPER',
 'DELTSAV',
 'TOTALSIMTIME',
 'TSMULT',
 'INATS',
 'TOTIMC',
 'DELT',
 'NSTP',
 'READNEWDATA',
 'KPER']

The scalar values 

In [84]:
mf6.simulation.TDIS.NPER

value:,np.int32(2)


and arrays:

In [85]:
mf6.simulation.TDIS.PERLEN

value:,"array([ 1., 300.])"
shape:,"(2,)"


The docstrings are extracted from the MODFLOW 6 source code from the used MODFLOW version as displayed with mf6.info.

The instance has many variables:

In [86]:
len(mf6.vars)

589

Filter for all TDIS entries:

In [87]:
{k: v for k, v in mf6.vars.items() if 'TDIS' in k}

{'TDIS/PERTIMSAV': array([0.]),
 'TDIS/TOPERTIM': array([0.]),
 'TDIS/TOTIMSAV': array([0.]),
 'TDIS/PERTIM': array([1.]),
 '__INPUT__/SIM/TDIS/NSTP': array([ 1, 20], dtype=int32),
 'TDIS/ITMUNI': array([4], dtype=int32),
 'TDIS/KSTP': array([1], dtype=int32),
 'TDIS/PERLEN': array([  1., 300.]),
 'TDIS/TOTIM': array([1.]),
 'TDIS/NPER': array([2], dtype=int32),
 'TDIS/DELTSAV': array([0.]),
 'TDIS/TOTALSIMTIME': array([301.]),
 '__INPUT__/SIM/TDIS/PERLEN': array([  1., 300.]),
 'TDIS/TSMULT': array([1., 1.]),
 'TDIS/INATS': array([0], dtype=int32),
 'TDIS/TOTIMC': array([0.]),
 'TDIS/DELT': array([1.]),
 'TDIS/NSTP': array([ 1, 20], dtype=int32),
 '__INPUT__/SIM/TDIS/TSMULT': array([1., 1.]),
 '__INPUT__/SIM/TDIS/NPER': array([2], dtype=int32),
 'TDIS/KPER': array([1], dtype=int32)}

Show the first 20:

In [88]:
dict(list(mf6.vars.items())[:20])

{'SLN_1/NCOL': array([10201], dtype=int32),
 'SLN_1/IMSLINEAR/ID': array([0, 0, 0, ..., 0, 0, 0], shape=(10201,), dtype=int32),
 '__INPUT__/GWF_WS_EXAMPLE_1/MODEL_SHAPE': array([  1, 101, 101], dtype=int32),
 'GWF_WS_EXAMPLE_1/CHD-1/ISADVPAK': array([0], dtype=int32),
 'GWF_WS_EXAMPLE_1/VSC/THERMIVISC': array([0], dtype=int32),
 'GWF_WS_EXAMPLE_1/INOC': array([1009], dtype=int32),
 'TDIS/PERTIMSAV': array([0.]),
 'GWF_WS_EXAMPLE_1/GNC/IOUT': array([1012], dtype=int32),
 'SLN_1/IMSLINEAR/WLU': array([0.]),
 'GWF_WS_EXAMPLE_1/HFB/NHFB': array([0], dtype=int32),
 '__INPUT__/GWF_WS_EXAMPLE_1/NPF/K': array([1., 1., 1., ..., 1., 1., 1.], shape=(10201,)),
 'GWF_WS_EXAMPLE_1/DIS/ANGROT': array([0.]),
 'GWF_WS_EXAMPLE_1/NPF/K33': array([0.3, 0.3, 0.3, ..., 0.3, 0.3, 0.3], shape=(10201,)),
 'SLN_1/IMSLINEAR/IW': array([0, 0, 0, ..., 0, 0, 0], shape=(10201,), dtype=int32),
 'GWF_WS_EXAMPLE_1/WEL-1/IVSC': array([0], dtype=int32),
 'SLN_1/IMSLINEAR/ARO': array([0.]),
 'GWF_WS_EXAMPLE_1/INEWTONUR': 

A simulation can have several models types. These include:

+ gwf6 - flow model

+ gwt6 - constituent transport model

+ gwe6 - energy transport model

They are available as a dictionary:

In [90]:
mf6.models.keys()

dict_keys(['gwf6'])

The example has only a flow models. Let’s get them:

In [93]:
flow_models = mf6.models['gwf6']

MODFLOW supports multiple flow models. An example is the nesting an inner model with finer discretization into an outer model withcoarse discritization. Let’s get the flow model names:

In [94]:
flow_models.keys()

dict_keys(['gwf_ws_example_1'])

This simple simulation has only one flow model. We get a reference to this model:

In [96]:
gwf = flow_models['gwf_ws_example_1']

We can get a list if all available attributes, removing all names beginning with an underscore because they are internal names or special methods:

In [97]:
[attr for attr in dir(gwf) if not attr.startswith('_')]

['X',
 'allow_convergence',
 'dis_name',
 'dis_type',
 'get_package',
 'kper',
 'kstp',
 'mf6',
 'name',
 'nodetouser',
 'nper',
 'nstp',
 'package_dict',
 'package_list',
 'package_names',
 'package_types',
 'packages',
 'shape',
 'size',
 'solution_id',
 'subcomponent_id',
 'totim',
 'usertonode']

All attributes are available via tab completion, i.e. typing <TAB> after the dot will show the list of available attributes and will narrow it down to match typed characters:

In [98]:
gwf.nper

np.int32(2)

The packages of a model are also available:

In [99]:
gwf.packages

,description,is_mutable
name,,
dis,DIS Package: DIS,False
vsc,VSC Package: VSC,True
buy,BUY Package: BUY,True
wel-1,WEL Package: WEL-1,True
chd-1,CHD Package: CHD-1,True
gnc,GNC Package: GNC,True
ic,IC Package: IC,False
sto,STO Package: STO,False
npf,NPF Package: NPF,False


The well boundary condition is potentially mutable. Let’s try to get a mutable version:

In [101]:
gwf.packages.mywell.as_mutable_bc()

AttributeError: No package "well".
Available packages are:
    dis
    vsc
    buy
    wel-1
    chd-1
    gnc
    ic
    sto
    npf
    hfb
    csub
    mvr

This doesn’t work yet, because there is no boundary condition in the first, steady-state, stress period.

We create a reference to the model loop:

In [102]:
loop = mf6.model_loop()

and progress to the start of the second stress period:

In [104]:
for model_step in loop:
    if gwf.kper > 0:
        break

Note: Remember that the stress period count is zero-based. Therefore, the stress period with the index 0 is the first.

Now, we are at the beginning of the second stress period:

In [105]:
model_step.state

<States.timestep_start: 3>

and can create a mutable version of our well boundary conditions:

In [109]:
gwf.packages.chd_0.as_mutable_bc()

AttributeError: No package "chd_0".
Available packages are:
    dis
    vsc
    buy
    wel-1
    chd-1
    gnc
    ic
    sto
    npf
    hfb
    csub
    mvr

In [110]:
mywell = gwf.packages.mywell.as_mutable_bc()

AttributeError: No package "mywell".
Available packages are:
    dis
    vsc
    buy
    wel-1
    chd-1
    gnc
    ic
    sto
    npf
    hfb
    csub
    mvr

There is one well in the middle of the model.

In [112]:
MF6?

Init signature:
MF6(
    sim_path,
    dll_path=None,
    use_modflow_api=True,
    advance_first_step=True,
    verbose=False,
    new_step_only=False,
    do_solution_loop=True,
    _develop=False,
)
Docstring:     
Wrapper around XmiWrapper and modflowapi.

`advance_first_step = True` progresses to the first model step with
model time > 0. This is needed to access any internal values of BCs.
File:           c:\users\lucialabarca\re-run noteboks\pymf6-validation\.pixi\envs\default\lib\site-packages\pymf6\mf6.py
Type:           type
Subclasses:     

We can progress to the next stress period:

In [113]:
for model_step in loop:
    if gwf.kper > 1:
        break

In [114]:
mywell

NameError: name 'mywell' is not defined

we can modify the pumping rate and make itr 10% higher: 

In [ ]:
mywell.q *= 1.1

The changes are written back into MODFLOW simulation and will be used until for the calculations until MODFLOW reads new values for this boundary condition. This happens at the start of the next stress period or when new time series values are read.

In [115]:
mywell

NameError: name 'mywell' is not defined

In [ ]:
model_step.state

In [ ]:
model_step.simulation_group.model_names

In [ ]:
model_step.simulation_group.kper

In [116]:
gwf

XMIError: BMI exception in get_var_rank (for variable GWF_WS_EXAMPLE_1/DIS/NODES): Message from MODFLOW 6 'BMI Error, unknown variable: NODES at GWF_WS_EXAMPLE_1/DIS'